# Predicting heart disease using machine learning

This notebook looks into using various Python-based machine learning and
data science libraries in an attempt to build a machine learning model
capable of predicting whether or not someone has heart disease based
on their medical attributes.

We'll take the following approach.

1. Problem definition
2. Data
3. lEvaluation
4. Features
5. Modelling
6. Experimentation

## 1. Problem Definition

In a statement

> Given clinical parameters about a patient, can we predict
> whether or not they have heart disease?

## 2. Data

We can consult [the original data set](https://archive.ics.uci.edu/dataset/45/heart+disease).

However, it is more convenient to consult [the same data set on Kaggle](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data).

Here is a data dictionary of the available data.

Column Descriptions:

1. id (Unique id for each patient)
2. age (Age of the patient in years)
3. origin (place of study)
4. sex (Male/Female)
5. cp chest pain type ([typical angina, atypical angina, non-anginal, asymptomatic])
6. trestbps resting blood pressure (resting blood pressure (in mm Hg on admission to the hospital))
7. chol (serum cholesterol in mg/dl)
8. fbs (if fasting blood sugar > 120 mg/dl)
9. restecg (resting electrocardiographic results)
   -- Values: [normal, stt abnormality, lv hypertrophy]
10. thalach: maximum heart rate achieved
11. exang: exercise-induced angina (True/ False)
12. oldpeak: ST depression induced by exercise relative to rest
13. slope: the slope of the peak exercise ST segment
14. ca: number of major vessels (0-3) colored by fluoroscopy
15. thal: [normal; fixed defect; reversible defect]
16. num: the predicted attribute

## 3. Evaluation

> If we can reach 95% accuracy at predicting whether or not a
> patient has heart disease during the proof of concept, we
> will pursue the project further.

## 4. Features

This is where you'll acquire different information about each of the fatures in your data.

**Create a data dictionary**

1. age - Age of the patient in years
2. sex - 1 = male; 0 = female
3. cp - chest pain type
    - Typical angina: chest pain related to decreased blood supply to the heart
    - Atypical angina: chest pain not related to the heart
    - Non-anginal: typical esophageal spasms (not heart related)
    - Asymptomatic: chest pain not showing signs of disease
4. trestbps - resting blood pressure (in mm Hg on admission to the hospital)
   anything above 130-1490 is typically cause for concern
5. chol - serum cholesterol in mg/dl
    - Serum = LDL + HDL + 0.2 * triglycerides
    - About 200 is cause for concern
6. fbs - fasting blood sugar > 120 mg/dl: 1 = true; 0 = false)
    - '> 126' might signal diabetes
7. restecg (resting electrocardiographic results)
    - 0: Nothing to note
    - 1: ST-T Wave abnormality
    - 2: Left ventricular hypertrophy
8. thalach: maximum heart rate achieved
9. exang: exercise-induced angina (1 = yes; 0 = no)
10. oldpeak: ST depression (heart potentially not getting enough oxygen)
    induced by exercise relative to rest
11. slope: the slope of the peak exercise ST segment
    - 0: Upsloping
    - 1: Flatsloping
    - 2. Downsloping
12. ca: number of major vessels (0-3) colored by fluoroscopy
13. thal: Thalium stress result
    - 1: Normal
    - 3: Normal
    - 6: Fixed defect
    - 7: reversible defect
14. target: Have disease or not (1 = yes; 0 = no)

## Preparing the tools

We will use:

- `pandas`
- `matplotlib`
- `numpy`

For data analysis and manipulation

In [ ]:
# Import all the tools we need

# Import cytoolz to simplify code
from cytoolz.curried import *

# Regular EDA (exploratory data analysis) and plotting libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# %matplotlib inline

# Models from Scikit-Learn
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# del Evaluations
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RandomizedSearchCV,
    GridSearchCV,
)
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    RocCurveDisplay,
)

### Load Data

In [ ]:
df = pd.read_csv('data/heart-disease.csv')
df

In [ ]:
df.shape

## Data Exploration (exploratory data analysis or EDA)

The goal here is to find out more about the data and become
a subject-matter expert on the data set with which you are
working.

Steps

1. What question(s) are you trying to solve
2. What kind of data do we have and how do we treat different types?
3. What is missing from the data and how do you deal with it?
4. Where are the outliers and why should you care about them?
5. How can you add, change, or remove features to get more out of your data?

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Let's start by looking at the `target` variable (the "label")
# We'll find out how many items exist of each class
df['target'].value_counts()

These results are fairly balanced between our two labels

In [ ]:
# Let's plot the `target` values
df['target'].value_counts().plot(kind='bar', color=['salmon', 'lightblue'])
plt.show()

In [ ]:
# What can we learn about other data?
df.info()

In [ ]:
# What data is missing?
df.isna().sum()

In [ ]:
# Additional descriptive information
df.describe()

### Heart Disease Frequency According to Sex

Remember we are **exploring** the data

In [ ]:
df['sex'].value_counts()

In [ ]:
# Compare target column with sex column
pd.crosstab(df['target'], df['sex'])

In [ ]:
heart_disease_count = len(df[df['target'] == 1])
participant_count = len(df)
heart_disease_count  / participant_count * 100

In [ ]:
# Create a plot of our crosstab
pd.crosstab(df['target'], df['sex']).plot(kind='bar',
                                          figsize=(10, 6),
                                          color=['salmon', 'lightblue'])
plt.title('Heart Disease Frequency by Sex')
plt.xlabel('0 - No Disease; 1 - Disease')
plt.ylabel('Count')
plt.legend(['Female', 'Male'])
plt.xticks(rotation=0)
plt.show()

In [ ]:
pd.crosstab(df['target'], df['cp'])

In [ ]:
pd.crosstab(df['target'], df['cp']).plot(
    kind='bar',
    figsize=(10, 6),
)
plt.xticks(rotation=0)
plt.legend([
    'Typical angina',
    'Atypical angina',
    'Non-anginal',
    'Asymptomatic',
])
plt.show()

In [ ]:
plt.hist(df[df['target'] == 0]['thalach'], label='No heart disease')
plt.hist(df[df['target'] == 1]['thalach'], label='With heart disease')
plt.legend()
plt.show()

In [ ]:
# `thalach` is maximum heart rate achieved
df['thalach'].value_counts()

### Age versus Max Heart Rate for Heart Disease

In [ ]:
# Create a figure with two plots for our independent variables
plt.figure(figsize=(10, 6))

# Scatter plot with positive examples only
plt.scatter(df.age[df.target == 1],
            df.thalach[df.target == 1],
            c='salmon')

# Scatter plat with zero examples only
plt.scatter(df.age[df.target == 0],
            df.thalach[df.target == 0],
            c='lightblue')

plt.title('Heart Disease Against Age and Max Heart Rate')
plt.xlabel('Age')
plt.ylabel('Heart Rate')
plt.legend(['With Heart Diseases', 'No Heart Disease'])

plt.show()

In [ ]:
# View the distribution of age with a histogram
df.age.plot.hist()

plt.show()

### Heart Disease Frequency for Chest Pain (Type)

Remember our data dictionary values for chest pain

cp - chest pain type

    - Typical angina: chest pain related to decreased blood supply to the heart
    - Atypical angina: chest pain not related to the heart
    - Non-anginal: typical esophageal spasms (not heart related)
    - Asymptomatic: chest pain not showing signs of disease


In [ ]:
pd.crosstab(df.cp, df.target)

In [ ]:
# Make the crosstab more visual
pd.crosstab(df.cp, df.target).plot(kind='bar',
                                   figsize=(10, 6),
                                   color=['salmon', 'lightblue'])

# Add some communication
plt.title('Heart Disease Frequency Per Chest Pain')
plt.xlabel('Chest Pain Type')
plt.ylabel('Amount')
plt.legend(['No Heart Disease', 'With Heart Diseases'])
plt.xticks(rotation=0,
           labels=['Typical Angina',
                   'Atypical Angina',
                   'Non-Anginal',
                   'Asymptomatic',],
           ticks=[0, 1, 2, 3])
plt.show()

In [ ]:
plt.hist(df[df['target'] == 0]['cp'], label='No heart disease')
plt.hist(df[df['target'] == 1]['cp'], label='With heart disease')
plt.legend()

plt.show()

In [ ]:
df.head()

In [ ]:
# Make a correlation matrix
df.corr()

In [ ]:
# Let's make our correlation matrix a little prettier
correlation_matrix = df.corr()

# Create a figure
fig, ax = plt.subplots(figsize=(15, 10))
ax = sns.heatmap(
    correlation_matrix,
    annot=True, # annotate the heat map
    linewidths=0.5,
    fmt='.2f', # two digits after decimal point
    # cmap='YlGnBu', # yellow-green-blue color map
    cmap='managua', # diverging color map
    # See [Matplotlib color maps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
    center=0, # The data is **not** symmetric around 0 but it could be
)

plt.show()

9. exang: exercise-induced angina (1 = yes; 0 = no)

In [ ]:
# Let's look at the relationship between `exang` and `target`
# Oun matrix indicates they are negatively correlated.
pd.crosstab(df.exang, df.target).plot(
    kind='bar',
    figsize=(10, 6),
    color=['salmon', 'lightblue']
)

# Add some communication
plt.title('Heart Disease Frequency Per Exercise Induced Angina')
plt.xlabel('Chest Pain Type')
plt.ylabel('Exercise Induced Angina')
plt.legend(['No Heart Disease', 'With Heart Diseases'])
plt.xticks(rotation=0,
           labels=['Yes',
                   'No',],
           ticks=[0, 1])
plt.show()

## 5.0 Modelling

In [ ]:
df.head()

In [ ]:
# Initialize our random number generator for reproducibility
rng = np.random.default_rng(seed=42)

In [ ]:
# Split data into features (X) and labels (y)
X = df.drop('target', axis=1)
y= df['target']

In [ ]:
X

In [ ]:
y.head(), y.tail()

In [ ]:
# Then split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.int32).max)
)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_train

In [ ]:
y_test

Now that we've got our data split into training and test sets,
it's time to build a machine learning model.

We'll train it (find the patterns) on the training set.

And we'll test it (use the patterns) on the test set.

What learning model(s) should we use?

Let's consult our [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html).

Our map seems to recommend (the video, actually):
- K-neighbors classifier
- SVC ensemble classifier

But we're missing one:
- Logistic Regression

We're going to try three different machine learning models

1. Logistic Regression
2. K-Nearest Neighbors Classifier
3. Random Forest Classifier

In [ ]:
# Put models in a dictionary
models = {
    'Logistic Regression': LogisticRegression(
        random_state=rng.integers(np.iinfo(np.int32).max),
    ),
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(
        random_state=rng.integers(np.iinfo(np.int32).max),
    ),
}

In [ ]:
# Create a function to fit and score models
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Fit and evaluate specified model
    :param model: A sklearn model
    :param X_train: training data (no labels)
    :param X_test: testing data (no labels)
    :param y_train: training labels
    :param y_test: test labels
    """
    model.fit(X_train, y_train)
    result = model.score(X_test, y_test)
    return result

In [ ]:
# Evaluate each model

# Capture the score for each model
def fit_and_score(models, X_train, X_test, y_train, y_test):
    """
    Fit and evaluate specified model
    :param models: A dictionary of model names and models
    :param X_train: training data (no labels)
    :param X_test: testing data (no labels)
    :param y_train: training labels
    :param y_test: test labels
    """
    return itemmap(
        lambda item: (item[0], evaluate_model(item[1], X_train, X_test, y_train, y_test)),
        models
    )

In [ ]:
fit_and_score(models, X_train, X_test, y_train, y_test)

In [ ]:
pipe(
    models,
    itemmap(
        lambda item: (item[0], evaluate_model(item[1], X_train, X_test, y_train, y_test)),
    ),
)

In [ ]:
model_scores = fit_and_score(
    models=models,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
model_scores

This calculation produces a warning.

In production code, we would attempt to remove this warning; however,
in our learning scenario, we will leave the code as is.

### (Visual) Model Comparison

In [ ]:
model_compare = pd.DataFrame(
    model_scores,
    index=['accuracy'],
)
model_compare.T.plot.bar()
plt.show()

Now we've got a baseline model... and we know a model's first
predictions aren't always what we should base our next steps on.
What should we do?

Let's look at the following

- Hyperparameter tuning
- Feature importance
- Confusion matrix
- Cross-validation
- Precision
- Recall
- F1 score
- Classification report
- ROC curve
- Area under the ROC curve (AUC)

## Hyperparameter Tuning (by hand)

In [ ]:
# Let's tune KNN

train_scores = []
test_scores = []

# Create a list of values for `n_neighbors` (the first parameter)
neighbors = range(1, 20 + 1) # From 1 to 20
knn = KNeighborsClassifier()

# Loop through different numbers of neighbors
for n in neighbors:
    knn.set_params(n_neighbors=n)

    # Fit the algorithm
    knn.fit(X_train, y_train)

    # Update the training scores list
    train_scores.append(knn.score(X_train, y_train))

    # Update the test scores list
    test_scores.append(knn.score(X_test, y_test)    )

In [ ]:
train_scores

In [ ]:
test_scores

In [ ]:
# How might we visualize these scores
plt.plot(neighbors, train_scores, label='Training')
plt.plot(neighbors, test_scores, label='Test')

plt.title('KNN Classifier - Score versus Neighbors')
plt.xlabel('Number of Neighbors')
plt.ylabel('Model Score')
plt.legend()

# Make it easier to identify maxima
plt.xticks(np.arange(1, 20 + 1, 1))

plt.show()

print(f'Maximum KNN score on the test data: {max(test_scores)*100:.2f}%')

## Hyperparameter tuning with `RandomizedSearchCV`

We're going to tune

- LogisticRegression
- RandomForestClassifier

Using `RandomSearchCV`

In [ ]:
np.logspace(-4, 4, 20)

In [ ]:
# Create a hyperparameter grid for `LogisticRegression`
log_reg_grid = {
    'C': np.logspace(-4, 4, 20),
    'solver': ['liblinear'],
}

# Create a hyperparameter grid for `RandomForestClassifier'
rf_grid = {
    'n_estimators': np.arange(10, 1000, 50),
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': np.arange(2, 20, 2),
    'min_samples_leaf': np.arange(1, 20, 2),
}

Now we've got hyperparameter grids set up for each of our models,
let's tune them using `RandomizedSearchCV''

In [ ]:
# Tune `LogisticRegression` model

rng = np.random.default_rng(seed=42)

# Set up random hyperparameter search for `LogisticRegression`
rs_log_reg = RandomizedSearchCV(
    LogisticRegression(random_state=rng.integers(np.iinfo(np.int32).max)),
    param_distributions=log_reg_grid,
    cv=5, # Remember, the higher the number, the longer the test
    n_iter=20,
    verbose=True,
    random_state=rng.integers(np.iinfo(np.int32).max)
)

# Fit the random hyperparameter search model for `LogisticRegression`
rs_log_reg.fit(X_train, y_train)

In [ ]:
# Find the best hyperparameters
rs_log_reg.best_params_

In [ ]:
rs_log_reg.score(X_test, y_test)

In [ ]:
model_scores

Now that we tuned `LogisticRegression`, let's do the same for
`RandomForestClassifier`.

In [ ]:
# Tune `RandomForestClassifier` model

rng = np.random.default_rng(seed=42)

# Set up random hyperparameter search for `RandomForestClassifier`
rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=rng.integers(np.iinfo(np.int32).max)),
    param_distributions=rf_grid,
    cv=5, # Remember, the higher the number, the longer the test
    n_iter=20,
    verbose=True,
    random_state=rng.integers(np.iinfo(np.int32).max)
)

# Fit the random hyperparameter search model for `RandomForestClassifier`
rs_rf.fit(X_train, y_train)

In [ ]:
# Find the best hyperparameters
rs_rf.best_params_

In [ ]:
# Evaluate the randomized search `RandomForestClassifier`  model
rs_rf.score(X_test, y_test)

In [ ]:
model_scores

Three ways to tune our model:

1. By hand
2. `RandomizedSearchCV`
3. `GridSearchCV`

In the video, the plan is to:

- Use `GridSearchCV` for the `LogisticRegression` model
- I may try to use `GridSearchCV` for the `RandomForestClassifier` model

## Hyperparameter Tuning with `GridSearchCV`

Since our `LogisticRegression` model provides the best scores so far,
at least in the video, we'll try to improve these scores
using `GridSearchCV`

In [ ]:
# Remember to reinitialize our random seed
rng = np.random.default_rng(seed=42)

# Different hyperparameters for our `LogisticRegression` model
log_reg_grid = {
    'C': np.logspace(-4, 4, 30),
    'solver': ['liblinear'],
}

gs_log_reg = GridSearchCV(
    LogisticRegression(random_state=rng.integers(np.iinfo(np.int32).max),),
    param_grid=log_reg_grid,
    cv=5,
    verbose=True,
)

gs_log_reg.fit(X_train, y_train)


In [ ]:
# Check the best parameters
gs_log_reg.best_params_

In [ ]:
# Evaluate the grid search on our test data
gs_log_reg.score(X_test, y_test)

In [ ]:
model_scores

Let's try additional tuning of the `RandomForestClassifier`

In [ ]:
np.random.default_rng(seed=42)

# Different hyperparameters for our `RandomForestClassifier` model
rf_gs_grid = {
    'n_estimators': np.arange(900, 920, 10),
    'max_depth': [9, 10, 11],
    'min_samples_split': np.arange(15, 17 + 1, 1),
    'min_samples_leaf': np.arange(18, 20 + 1, 1),
}

rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=rng.integers(np.iinfo(np.int32).max),),
    param_grid=rf_gs_grid,
    cv=5,
    verbose=True,
)

rf_gs.fit(X_train, y_train)


In [ ]:
rf_gs.best_params_

In [ ]:
rf_gs.score(X_test, y_test)

In [ ]:
model_scores

In [ ]:
rs_rf.score(X_test, y_test)

## Evaluating our tuned machine learning classifier - beyond accuracy

We'll look at the following:

- ROC curve and AUC score
- Confusion matrix
- Classification report
- Precision
- Recall
- F1 score

...and it would be great to use cross-validation where possible

To make comparisons and evaluate our trained model, we must begin with **predictions**


In [ ]:
# We predict from the `X_test` data using our trained model
y_preds = gs_log_reg.predict(X_test)

In [ ]:
y_preds

In [ ]:
y_test

In [ ]:
# Plot ROC curve and calculate AUC
RocCurveDisplay.from_predictions(y_test, y_preds)

In [ ]:
RocCurveDisplay.from_estimator(gs_log_reg, X_test, y_test)
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(rs_rf, X_test, y_test)
plt.show()

In [ ]:
# Confusion matrix
print(confusion_matrix(y_test, y_preds))

In [ ]:
# Import seaborn
import seaborn as sns
sns.set(font_scale=1.5) # increase font size

def plot_conf_mat(y_test, y_preds):
    """
    Plot the confusion matrix using seaborn's `heatmap`
    :param y_test: The test labels
    :param y_preds: The predicted labels
    :return: None
    """
    fig, ax = plt.subplots(figsize=(3, 3))
    ax = sns.heatmap(confusion_matrix(y_test, y_preds),
                     annot=True, # annotate the boxes
                     cbar=False,
    )

    plt.xlabel('Predicted label')
    plt.ylabel('True label')

    plt.show()

In [ ]:
# Remember we are using the **corrected** implementation
# of `plot_conf_mat()`
plot_conf_mat(y_test, y_preds)

Now, we've got

- An ROC curve
- An AUC metric
- A confusion matrix.

Let's get:

- A classification report
- Cross-validated
    - Precision
    - Recall
    - F1 score(s)


In [ ]:
print(classification_report(y_test, y_preds))

### Calculate evaluation metrics using cross-validation

We're going to calculate:

- Accuracy
- Precision
- Recall
- F1 score

Using cross-validation.

To accomplish that goal, we'll use `cross_val_score()`.

We will calculate our different metrics using the `scoring` parameter
of `cross_val_score()`.

In [ ]:
# Check best hyperparameters for best model
# - Logistic Regression in the video
# - Random Forest Classifier in this workbook
gs_log_reg.best_params_

In [ ]:
# Create a new classifier with the best parameters
log_reg_clf = LogisticRegression(C=1.3738237958832638, solver='liblinear')

In [ ]:
# Cross-validated accuracy
log_reg_cv_acc = cross_val_score(
    log_reg_clf, X_train, y_train, cv=5, scoring='accuracy'
)
log_reg_cv_acc

In [ ]:
log_reg_cv_acc = np.mean(log_reg_cv_acc)
log_reg_cv_acc

In [ ]:
# Cross-validated precision
log_reg_cv_precision = cross_val_score(
    log_reg_clf, X_train, y_train, cv=5, scoring='precision'
)
log_reg_cv_precision

In [ ]:
log_reg_cv_precision = np.mean(log_reg_cv_precision)
log_reg_cv_precision

In [ ]:
# Cross-validated recall
log_reg_cv_recall = cross_val_score(
    log_reg_clf, X_train, y_train, cv=5, scoring='recall'
)
log_reg_cv_recall

In [ ]:
log_reg_cv_recall = np.mean(log_reg_cv_recall)
log_reg_cv_recall

In [ ]:
# Cross-validated F! score
log_reg_cv_f1 = cross_val_score(
    log_reg_clf, X_train, y_train, cv=5, scoring='f1'
)
log_reg_cv_f1

In [ ]:
log_reg_cv_f1 = np.mean(log_reg_cv_f1)
log_reg_cv_f1

In [ ]:
cv_metrics = pd.DataFrame(
    {
        'Accuracy': log_reg_cv_acc,
        'Precision': log_reg_cv_precision,
        'Recall': log_reg_cv_recall,
        'F1': log_reg_cv_f1,
    },
    index=[0]
)

cv_metrics.T.plot.bar(title='Cross-validated classification metrics',
                      legend=False,)
plt.show()